In [ ]:
"""
DATASET VERIFICATION SCRIPT
============================
Run this FIRST in Colab before any experiment.
Prints every number cited in the paper.
All output verified 2026-05-16.

Expected output is shown in comments below each assertion.
If any assertion fails, the dataset may have changed — stop and investigate.
"""

import json
from collections import Counter, defaultdict

# DATASET_PATH = '/content/indian_bail_judgments.json'
DATASET_PATH = '/content/indian_bail_judgments.json'

with open(DATASET_PATH) as f:
    data = json.load(f)

print("=" * 70)
print("DATASET VERIFICATION — IndianBailJudgments-1200")
print("=" * 70)

# ── 1. Basic ──────────────────────────────────────────────────
total    = len(data);                assert total == 1200
granted  = sum(1 for c in data if c['bail_outcome'] == 'Granted');  assert granted == 736
rejected = total - granted;          assert rejected == 464
print(f"[1]  Total cases:              {total}")
print(f"[2]  Fields per case:          {len(data[0].keys())} fields")
print(f"[3]  Granted:                  {granted} ({granted/total*100:.1f}%)")
print(f"[3]  Rejected:                 {rejected} ({rejected/total*100:.1f}%)")

# ── 2. Parity ─────────────────────────────────────────────────
pt = [c for c in data if c.get('parity_argument_used') == True]
pf = [c for c in data if c.get('parity_argument_used') == False]
assert len(pt) == 341
assert len(pf) == 859
pt_g = sum(1 for c in pt if c['bail_outcome'] == 'Granted')  # 263
pf_g = sum(1 for c in pf if c['bail_outcome'] == 'Granted')  # 473
pt_r = pt_g/len(pt)*100   # 77.1
pf_r = pf_g/len(pf)*100   # 55.1
print(f"\n[4]  Parity=True:              {len(pt)} ({len(pt)/total*100:.1f}%)")
print(f"[5]  Parity=False:             {len(pf)} ({len(pf)/total*100:.1f}%)")
print(f"[6]  Grant rate parity=True:   {pt_r:.1f}% (n={len(pt)}, granted={pt_g})")
print(f"[7]  Grant rate parity=False:  {pf_r:.1f}% (n={len(pf)}, granted={pf_g})")
print(f"[8]  Parity premium:           +{pt_r-pf_r:.1f} pp")

# ── 3. Keywords ───────────────────────────────────────────────
KWS = ['co-accused', 'parity', 'similarly placed']
def text(c):
    li = c.get('legal_issues','')
    if isinstance(li,list): li = ' '.join(li)
    return (str(c.get('facts',''))+' '+str(c.get('judgment_reason',''))+' '+str(li)).lower()
pt_kw = sum(1 for c in pt if any(k in text(c) for k in KWS))
pf_kw = sum(1 for c in pf if any(k in text(c) for k in KWS))
assert pt_kw == 291
print(f"\n[9]  Keyword coverage parity=True:   {pt_kw}/{len(pt)} = {pt_kw/len(pt)*100:.1f}%")
print(f"[10] Keyword in parity=False cases:  {pf_kw}/{len(pf)} = {pf_kw/len(pf)*100:.1f}% (false positives)")

# ── 4. Demographics ───────────────────────────────────────────
gc = Counter(c.get('accused_gender','Unknown') for c in data)
print(f"\n[12] Gender: Male={gc['Male']}, Female={gc['Female']}, Unknown={gc['Unknown']}, Multiple={gc.get('Multiple',0)}")
for g in ['Male','Female']:
    gl = [c for c in data if c.get('accused_gender')==g]
    gr = sum(1 for c in gl if c['bail_outcome']=='Granted')
    print(f"[13] Grant rate {g}: {gr/len(gl)*100:.1f}%")
fem_pt = [c for c in pt if c.get('accused_gender')=='Female']
mal_pt = [c for c in pt if c.get('accused_gender')=='Male']
print(f"[14] Female parity=True:  n={len(fem_pt)}, grant={sum(1 for c in fem_pt if c['bail_outcome']=='Granted')/len(fem_pt)*100:.1f}%")
print(f"[16] Male   parity=True:  n={len(mal_pt)}, grant={sum(1 for c in mal_pt if c['bail_outcome']=='Granted')/len(mal_pt)*100:.1f}%")

# ── 5. Religious names ────────────────────────────────────────
HM = ['sharma','singh','kumar','gupta','verma','patel','shah','yadav','mishra',
      'tiwari','pandey','dubey','rajput','joshi','mehta','ram','lal','das']
MM = ['khan','mohammed','mohammad','shaikh','sheikh','ansari','qureshi','siddiqui',
      'ali','hussain','hassan','malik','pasha','mirza','beg','ahmed','ahmad']
def ih(c): n=str(c.get('accused_name','')).lower(); return any(m in n for m in HM)
def im(c): n=str(c.get('accused_name','')).lower(); return any(m in n for m in MM)
h_all = sum(1 for c in data if ih(c)); assert h_all == 468
m_all = sum(1 for c in data if im(c)); assert m_all == 163
h_pt  = sum(1 for c in pt if ih(c));   assert h_pt  == 145
m_pt  = sum(1 for c in pt if im(c));   assert m_pt  == 49
print(f"\n[22] Hindu-identifiable all:     {h_all} ({h_all/total*100:.1f}%)")
print(f"[23] Muslim-identifiable all:    {m_all} ({m_all/total*100:.1f}%)")
print(f"[24] Hindu parity=True:          {h_pt}")
print(f"[25] Muslim parity=True:         {m_pt}")

# ── 6. Temporal ───────────────────────────────────────────────
years = [int(str(c.get('date','0000'))[:4]) for c in data
         if str(c.get('date','0000'))[:4].isdigit()]
years = [y for y in years if 1970 <= y <= 2030]
print(f"\n[18] Year range: {min(years)}–{max(years)}")
by_dec = defaultdict(list)
for c in data:
    try:
        y = int(str(c.get('date',''))[:4])
        if 1970<=y<=2030: by_dec[(y//10)*10].append(c)
    except: pass
print("[19-21] Decades:")
for dec in sorted(by_dec):
    cases = by_dec[dec]
    pt_d  = sum(1 for c in cases if c.get('parity_argument_used')==True)
    gr_d  = sum(1 for c in cases if c['bail_outcome']=='Granted')
    print(f"  {dec}s: n={len(cases):4d}  parity={pt_d:3d} ({pt_d/len(cases)*100:5.1f}%)  grant={gr_d/len(cases)*100:5.1f}%")

# ── 7. Crime types ────────────────────────────────────────────
print("\n[31] Crime types:")
ct = defaultdict(lambda:{'n':0,'g':0,'pt':0})
for c in data:
    t=c.get('crime_type','Unknown'); ct[t]['n']+=1
    if c['bail_outcome']=='Granted': ct[t]['g']+=1
    if c.get('parity_argument_used')==True: ct[t]['pt']+=1
print(f"  {'Crime':<22}  {'N':>5}  {'Grant%':>7}  {'Parity%':>8}")
for t,d in sorted(ct.items(),key=lambda x:-x[1]['n']):
    print(f"  {t:<22}  {d['n']:>5}  {d['g']/d['n']*100:>6.1f}%  {d['pt']/d['n']*100:>7.1f}%")

# ── Final Summary ─────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL — Numbers for paper (all assertions passed ✓)")
print("="*70)
print(f"Total cases:              1200")
print(f"Parity=True:              341 (28.4%)")
print(f"Parity=False:             859 (71.6%)")
print(f"Grant rate (all):         61.3%")
print(f"Grant rate parity=True:   77.1% (n=341, granted=263)")
print(f"Grant rate parity=False:  55.1% (n=859, granted=473)")
print(f"Parity premium:           +22.1 pp")
print(f"Keyword coverage:         291/341 = 85.3%")
print(f"False positive keyword:   163/859 = 19.0%")
print(f"Year range:               1975–2025")
print(f"Male accused:             1071  Female: 123")
print(f"Hindu-identifiable:       468 all / 145 parity=True")
print(f"Muslim-identifiable:      163 all / 49  parity=True")
print(f"Female parity=True:       33  grant rate 90.9%")
print(f"Male   parity=True:       307 grant rate 75.6%")
print(f"\nAll assertions passed ✓  Dataset is clean and ready.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')